In [1]:
from pathlib import Path

import polars as pl

ROOT = Path.cwd()
OUT_DIR = ROOT / "data" / "output"
if not OUT_DIR.exists():
    OUT_DIR = ROOT.parent / "data" / "output"

parquet_path = OUT_DIR / "dashboard" / "metrics.parquet"
if not parquet_path.exists():
    raise FileNotFoundError(f"Parquet file not found: {parquet_path}")

raw_df = pl.read_parquet(parquet_path)

In [2]:
raw_df.head()

unique_id,wmape,sum_y,n_points,n_with_sales,unidad,n_spine,seccion,store,sku,node_kind
str,f64,f64,u32,u32,str,u32,str,str,str,str
"""1||T:00003||S:152629""",0.69708,13234.53,731,24,"""Valor ($)""",731,"""1""","""00003""","""152629""","""tienda_sku"""
"""1||T:00003||S:415256""",1.0,1.0,731,1,"""Unidades""",731,"""1""","""00003""","""415256""","""tienda_sku"""
"""1||S:589506""",0.880142,3795.0,0,13,"""Valor ($)""",null,"""1""",null,"""589506""","""sku"""
"""1||T:00001||S:419098""",0.869554,9164.94,731,22,"""Valor ($)""",731,"""1""","""00001""","""419098""","""tienda_sku"""
"""23||T:00046||S:514456""",1.0,1.0,624,1,"""Unidades""",624,"""23""","""00046""","""514456""","""tienda_sku"""


In [3]:
best_sku = {
    sec: {
        sto: (
            raw_df.filter(
                (pl.col("node_kind") == "tienda_sku")
                & (pl.col("seccion") == sec)
                & (pl.col("store") == sto)
                & (pl.col("unidad") == "Valor ($)")
                & pl.col("wmape").is_not_null()
                & (pl.col("wmape") != 0)
            )
            .sort("wmape")
            .get_column("sku")
            .head(50)
            .to_list()
        )
        for sto in (
            raw_df.filter(
                (pl.col("node_kind") == "tienda")
                & (pl.col("seccion") == sec)
                & (pl.col("unidad") == "Valor ($)")
            )
            .sort("wmape")
            .get_column("store")
            .head(5)
            .to_list()
        )
    }
    for sec in ["1", "23"]
}

best_sku

{'1': {'00001': ['39472',
   '580661',
   '459977',
   '541427',
   '516028',
   '567332',
   '41796',
   '46499',
   '46853',
   '75048',
   '208',
   '483046',
   '64048',
   '58905',
   '2322',
   '218557',
   '43549',
   '190454',
   '478352',
   '447684',
   '42231',
   '127359',
   '565118',
   '6656',
   '61722',
   '46713',
   '299478',
   '118209',
   '478353',
   '379145',
   '406342',
   '447627',
   '406018',
   '145082',
   '43550',
   '152629',
   '1063',
   '605956',
   '127360',
   '133336',
   '541426',
   '1760',
   '39591',
   '41542',
   '31894',
   '500770',
   '38333',
   '57488',
   '570746',
   '55921'],
  '00003': ['581783',
   '106356',
   '58905',
   '143871',
   '85860',
   '39472',
   '168765',
   '473932',
   '116741',
   '64048',
   '46499',
   '127359',
   '124382',
   '332817',
   '31894',
   '169938',
   '495934',
   '478352',
   '492075',
   '101740',
   '495933',
   '494947',
   '477265',
   '555434',
   '101743',
   '565871',
   '145082',
   '51782'

In [4]:
best_skus = {}
for sec in ["1", "23"]:
    best_skus.update(best_sku[sec])
best_skus

{'00001': ['39472',
  '580661',
  '459977',
  '541427',
  '516028',
  '567332',
  '41796',
  '46499',
  '46853',
  '75048',
  '208',
  '483046',
  '64048',
  '58905',
  '2322',
  '218557',
  '43549',
  '190454',
  '478352',
  '447684',
  '42231',
  '127359',
  '565118',
  '6656',
  '61722',
  '46713',
  '299478',
  '118209',
  '478353',
  '379145',
  '406342',
  '447627',
  '406018',
  '145082',
  '43550',
  '152629',
  '1063',
  '605956',
  '127360',
  '133336',
  '541426',
  '1760',
  '39591',
  '41542',
  '31894',
  '500770',
  '38333',
  '57488',
  '570746',
  '55921'],
 '00003': ['581783',
  '106356',
  '58905',
  '143871',
  '85860',
  '39472',
  '168765',
  '473932',
  '116741',
  '64048',
  '46499',
  '127359',
  '124382',
  '332817',
  '31894',
  '169938',
  '495934',
  '478352',
  '492075',
  '101740',
  '495933',
  '494947',
  '477265',
  '555434',
  '101743',
  '565871',
  '145082',
  '51782',
  '40399',
  '108193',
  '6656',
  '494254',
  '817',
  '565117',
  '127360',
  '

In [5]:
sales_path = OUT_DIR / "sales.parquet"
sales_df = pl.read_parquet(sales_path)
sales_df["STORE_ID", "SKU_ID"].unique()

STORE_ID,SKU_ID
str,str
"""00122""","""589459"""
"""00022""","""383416"""
"""00012""","""34644"""
"""00006""","""474063"""
"""00122""","""330384"""
…,…
"""00154""","""453737"""
"""00025""","""403987"""
"""00122""","""568160"""


In [6]:
best_skus_df = pl.DataFrame(
    [
        {"STORE_ID": sto, "SKU_ID": sku}
        for sto, skus in best_skus.items()
        for sku in skus
    ]
).cast(
    {
        "STORE_ID": pl.String,
        "SKU_ID": pl.String,
    }
)

sales_df.join(
    best_skus_df,
    on=["STORE_ID", "SKU_ID"],
    how="inner",
)

SALES_DATE,SKU_ID,STORE_ID,TRAN_TYPE,SLS_VAL,SLS_QTY,RTRN_QTY,RTRN_VAL
datetime[μs],str,str,str,f64,f64,f64,f64
2025-12-15 00:00:00,"""543149""","""00005""","""P""",264.1,2.0,0.0,0.0
2025-08-14 00:00:00,"""459977""","""00001""","""R""",2973.47,15.0,0.0,0.0
2025-11-03 00:00:00,"""332817""","""00003""","""P""",750.0,10.0,0.0,0.0
2025-02-24 00:00:00,"""159124""","""00063""","""P""",150.0,5.0,0.0,0.0
2024-01-12 00:00:00,"""450298""","""00005""","""P""",1543.5,56.0,0.0,0.0
…,…,…,…,…,…,…,…
2024-08-04 00:00:00,"""99039""","""00046""","""R""",387.0,3.0,0.0,0.0
2025-10-06 00:00:00,"""218557""","""00001""","""R""",3205.63,27.0,0.0,0.0
2025-09-30 00:00:00,"""68909""","""00005""","""P""",658.23,14.0,0.0,0.0


In [7]:
best_sku = {}

sec = "1"

stos = (
    raw_df.filter(
        (pl.col("node_kind") == "tienda")
        & (pl.col("seccion") == sec)
        & (pl.col("unidad") == "Valor ($)")
    )
    .sort("wmape")
    .get_column("store")
    .head(5)
    .to_list()
)

skus = (
    raw_df.filter(
        (pl.col("node_kind") == "tienda_sku")
        & (pl.col("seccion") == sec)
        & (pl.col("unidad") == "Valor ($)")
        & pl.col("store").is_in(stos)
        & pl.col("wmape").is_not_null()
        & (pl.col("wmape") != 0)
        & (pl.col("n_points") == 28)
    )
    .group_by("sku")
    .agg(
        pl.col("store").n_unique().alias("n_stores"),
        pl.col("wmape").mean().alias("wmape_mean"),
    )
    .filter(pl.col("n_stores") == len(stos))
    .sort("wmape_mean")
    .head(50)
    .get_column("sku")
    .to_list()
)

skus

[]

In [8]:
selected_stores = {
    sec: (
        raw_df.filter(
            (pl.col("node_kind") == "tienda")
            & (pl.col("seccion") == sec)
            & (pl.col("unidad") == "Valor ($)")
            & pl.col("wmape").is_not_null()
            & (pl.col("wmape") != 0)
        )
        .sort("wmape")
        .get_column("store")
        .head(5)
        .to_list()
    )
    for sec in ["1", "23"]
}
selected_stores

{'1': ['00001', '00003', '00211', '00154', '00063'],
 '23': ['00006', '00012', '00005', '00046', '00095']}

In [9]:
best_sku = []

for sec, stores in selected_stores.items():
    sku_candidates = (
        raw_df.filter(
            (pl.col("node_kind") == "tienda_sku")
            & (pl.col("seccion") == sec)
            & (pl.col("unidad") == "Valor ($)")
            & pl.col("store").is_in(stores)
            & pl.col("wmape").is_not_null()
            & (pl.col("wmape") != 0)
            & (pl.col("n_points") >= 20)
        )
        .sort("wmape")
        .head(50)
        .get_column("sku")
        .to_list()
    )

    best_sku += sku_candidates

best_sku

['412091',
 '278120',
 '39472',
 '581783',
 '580661',
 '459977',
 '541427',
 '106356',
 '495933',
 '516028',
 '568160',
 '567332',
 '41796',
 '46499',
 '46853',
 '75048',
 '58905',
 '143871',
 '85860',
 '208',
 '483046',
 '64048',
 '606556',
 '39472',
 '568161',
 '58905',
 '2322',
 '568160',
 '489263',
 '218557',
 '43549',
 '190454',
 '168765',
 '478352',
 '565871',
 '447684',
 '42231',
 '473932',
 '456855',
 '43422',
 '568160',
 '127359',
 '565118',
 '6656',
 '61722',
 '116741',
 '100622',
 '46713',
 '299478',
 '495933',
 '603595',
 '195295',
 '219991',
 '587387',
 '113618',
 '514081',
 '582947',
 '333113',
 '113618',
 '501237',
 '113618',
 '218058',
 '103986',
 '71765',
 '113618',
 '448634',
 '32757',
 '71765',
 '45642',
 '197305',
 '474665',
 '28303',
 '333110',
 '197305',
 '306915',
 '593461',
 '306915',
 '5898',
 '598287',
 '218058',
 '392615',
 '600005',
 '582946',
 '45642',
 '385301',
 '519600',
 '448849',
 '17245',
 '25879',
 '126196',
 '523647',
 '288718',
 '466270',
 '419464'

In [ ]:
parquet_path = OUT_DIR / "forecast.parquet"
if not parquet_path.exists():
    raise FileNotFoundError(f"Parquet file not found: {parquet_path}")

raw_df = pl.read_parquet(parquet_path)

In [ ]:
raw_df